# Crawler
Spider + Playwright

In [1]:
!pip install scrapy scrapy-playwright playwright playwright-stealth nest-asyncio -q
!playwright install chromium
!playwright install-deps chromium
import nest_asyncio
nest_asyncio.apply()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 kB 930.6 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 352.5/352.5 kB 7.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 39.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 79.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 3.4 MB/s eta 0:00:00
(node:156) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation 

In [2]:
!playwright --version

Version 1.58.0


In [3]:
import scrapy
from scrapy_playwright.page import PageMethod
from scrapy.crawler import CrawlerProcess
from scrapy.utils.project import get_project_settings
from datetime import datetime

In [4]:
raw_items: list[dict] = []

In [5]:
%%writefile settings.py

DOWNLOAD_HANDLERS = {
    "http": "scrapy_playwright.handler.ScrapyPlaywrightDownloadHandler",
    "https": "scrapy_playwright.handler.ScrapyPlaywrightDownloadHandler",
}

# Reactor is required to use Playwright with Scrapy
TWISTED_REACTOR = "twisted.internet.asyncioreactor.AsyncioSelectorReactor"

PLAYWRIGHT_LAUNCH_OPTIONS = {
    "headless": True, # Change to False if want to see the browser in action
    "args": [
        "--no-sandbox",
        "--disable-setuid-sandbox",
        "--disable-dev-shm-usage",
        "--disable-blink-features=AutomationControlled",
    ],
}

# Simulate screen parameters and User-Agent
PLAYWRIGHT_CONTEXT_ARGS = {
    "user_agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "viewport": {"width": 1920, "height": 1080},
}

ROBOTSTXT_OBEY = False
CONCURRENT_REQUESTS = 2
DOWNLOAD_DELAY = 3
TELNETCONSOLE_ENABLED = False # Turn off to avoid port errors on Notebook

Writing settings.py


In [6]:
class ItViecSpider(scrapy.Spider):
    name = "itviec"

    custom_settings = {
        "TWISTED_REACTOR": "twisted.internet.asyncioreactor.AsyncioSelectorReactor",
        "DOWNLOAD_HANDLERS": {
            "https": "scrapy_playwright.handler.ScrapyPlaywrightDownloadHandler",
            "http": "scrapy_playwright.handler.ScrapyPlaywrightDownloadHandler",
        },
        "PLAYWRIGHT_LAUNCH_OPTIONS": {"headless": True},
        "USER_AGENT": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "ROBOTSTXT_OBEY": False,
        "CONCURRENT_REQUESTS": 2,
        "DOWNLOAD_DELAY": 2,
    }

    def start_requests(self):
        base_url = "https://itviec.com/it-jobs"
        for page in range(1, 16):
            url = f"{base_url}?page={page}"
            yield scrapy.Request(
                url=url,
                meta={
                    "playwright": True,
                    "playwright_page_methods": [
                        PageMethod("wait_for_selector", "h3[data-search--job-selection-target='jobTitle']"),
                    ],
                },
                callback=self.parse
            )

    def parse(self, response):
        job_title_elements = response.css("h3[data-search--job-selection-target='jobTitle']")
        self.logger.info(f"Tìm thấy {len(job_title_elements)} jobs tại {response.url}")

        for element in job_title_elements:
            job_url = element.css("::attr(data-url)").get()
            title   = element.css("::text").get(default="").strip()

            if job_url:
                yield scrapy.Request(
                    url=response.urljoin(job_url),
                    meta={
                        "playwright": True,
                        "playwright_page_methods": [
                            PageMethod("wait_for_selector", ".job-show-info", timeout=15000),
                        ],
                        "title":        title,
                    },
                    callback=self.parse_job_details
                )

    def parse_job_details(self, response):
        # Company name
        company_name = response.css("div[class='employer-name']::text").get(default="").strip()

        # Title
        title = response.meta.get("title", "N/A")

        # Preview job
        job_header = response.xpath("//div[contains(@class, 'job-show-info')]")
        info_container = job_header.xpath("./div[contains(@class, 'imb-3')]")

        # Location
        location = info_container.xpath(".//svg[use[contains(@href, 'map-pin')]]/following-sibling::span/text()").get(default="").strip()

        # Work Mode 
        work_mode = info_container.xpath(".//div[contains(@class, 'preview-header-item')]//span[not(contains(text(), 'Posted'))]/text()").get(default="").strip()

        # Posted time
        posted_at = info_container.xpath(".//svg[use[contains(@href, 'clock')]]/following-sibling::span/text()").get(default="").strip()
        posted_at = posted_at.replace("Posted", "").strip()

        # Skills
        skills = info_container.xpath(".//div[contains(text(), 'Skills:')]/following-sibling::div/a/text()").getall()
        skills = [s.strip() for s in skills]

        # Job expertise
        job_expertise = info_container.xpath(".//div[contains(text(), 'Job Expertise:')]/following-sibling::div/a/text()").getall()
        job_expertise = [e.strip() for e in job_expertise]

        # Job domain
        job_domains = info_container.xpath(".//div[contains(text(), 'Job Domain:')]/following-sibling::div//div[contains(@class, 'itag')]/text()").getall()
        job_domains = [d.strip() for d in job_domains if d.strip()]

        # Job detail
        content_section = response.xpath("//section[contains(@class, 'job-content')]")

        # Top 3 reasons to join us 
        reasons = content_section.xpath(".//div[h2[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'reasons')]]//li/text()").getall()
        reasons = [r.strip() for r in reasons]

        # Job description
        job_description_raw = content_section.xpath(".//div[h2[contains(., 'Job description')]]//*[not(self::h2)]//text()").getall()
        job_description = "\n".join([t.strip() for t in job_description_raw if t.strip()])

        # Your skills and experience
        skills_experience_raw = content_section.xpath(".//div[h2[contains(., 'skills and experience')]]//*[not(self::h2)]//text()").getall()
        skills_experience = "\n".join([t.strip() for t in skills_experience_raw if t.strip()])

        # Why you'll love working here
        why_love_raw = content_section.xpath(".//div[h2[contains(., 'love working here')]]//*[not(self::h2)]//text()").getall()
        why_love = "\n".join([t.strip() for t in why_love_raw if t.strip()])

        # Company information
        employer_section = response.xpath("//section[contains(@class, 'job-show-employer-info')]")

        # Company full name
        company_full_name = employer_section.xpath(".//div[contains(@class, 'imt-5')]/p/text()").get(default="").strip()

        # Company type
        company_type = employer_section.xpath(".//div[div[contains(text(), 'Company type')]]/div[contains(@class, 'text-end')]/text()").get(default="").strip()

        # Company industry 
        industry = "".join(employer_section.xpath(".//div[div[contains(text(), 'Company industry')]]/div[contains(@class, 'text-end')]//text()").getall()).strip()

        # Company size 
        company_size = " ".join(employer_section.xpath(".//div[div[contains(text(), 'Company size')]]/div[contains(@class, 'text-end')]//text()").getall()).split()
        company_size = " ".join(company_size)

        # Country 
        country = employer_section.xpath(".//div[div[contains(text(), 'Country')]]//span/text()").get(default="").strip()

        # Working days
        working_days = employer_section.xpath(".//div[div[contains(text(), 'Working days')]]/div[contains(@class, 'text-end')]/text()").get(default="").strip()

        # Overtime policy
        overtime_policy = employer_section.xpath(".//div[div[contains(text(), 'Overtime policy')]]/div[contains(@class, 'text-end')]/text()").get(default="").strip()

        yield {
            "job_url": response.url,
            "title": title,
            "location": location,
            "work_mode": work_mode,
            "posted_at": posted_at,
            "skills": skills,
            "job_expertise": job_expertise,
            "job_domains": job_domains,
            "top_reasons": reasons,
            "description": job_description,
            "requirements": skills_experience,
            "benefits_description": why_love,
            "company_name": company_name,
            "company_full_name": company_full_name,
            "type": company_type,
            "industry": industry,
            "size": company_size,
            "country": country,
            "working_days": working_days,
            "overtime_policy": overtime_policy,
            "crawled_at":   datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
        }

In [7]:
raw_items.clear()

class InMemoryPipeline:
    """Does NOT write any file. Purely collects items into raw_items."""

    def process_item(self, item, spider):
        raw_items.append(dict(item))
        return item

process = CrawlerProcess(settings={
    "TWISTED_REACTOR": "twisted.internet.asyncioreactor.AsyncioSelectorReactor",
    "ITEM_PIPELINES": {"__main__.InMemoryPipeline": 100},
    "LOG_LEVEL": "ERROR",  
})
process.crawl(ItViecSpider)
process.start()  # blocks until all requests finish

print(f"\n Crawl complete — {len(raw_items)} raw items collected in memory")


 Crawl complete — 300 raw items collected in memory


# Parser
Pydantic Schema

In [8]:
import json
import re
from datetime import date, datetime, timedelta
from typing import List, Optional
from pydantic import BaseModel, Field, field_validator

import json
import re
from datetime import date, datetime, timedelta
from typing import List, Optional

In [9]:
_JOB_ID_RE = re.compile(r"-([A-Za-z]?\d+)$")

_RELATIVE_OFFSETS: list[tuple[re.Pattern, str, int]] = [
    (re.compile(r"(\d+)\s+hour",  re.I), "hours",  0),
    (re.compile(r"(\d+)\s+day",   re.I), "days",   1),
    (re.compile(r"(\d+)\s+week",  re.I), "weeks",  7),
    (re.compile(r"(\d+)\s+month", re.I), "months", 30),
]


def _extract_job_id(url: str) -> Optional[str]:
    """Extract a short unique ID from the job URL slug, e.g. 'itviec-0458'."""
    if not url:
        return None
    clean = url.split("?")[0].split("#")[0].rstrip("/")
    slug  = clean.split("/")[-1] if "/" in clean else clean
    m = _JOB_ID_RE.search(slug)
    return f"itviec-{m.group(1)}" if m else f"itviec-{slug}"


def _parse_posted_date(posted_at: str, crawled_at: str) -> Optional[date]:
    """
    Convert a relative string ('5 hours ago', '3 days ago', …) to a
    calendar date by subtracting the offset from the crawl timestamp.
    Returns None when parsing fails.
    """
    try:
        base = datetime.strptime(crawled_at, "%Y-%m-%d %H:%M:%S")
    except (ValueError, TypeError):
        return None

    text = (posted_at or "").lower()

    if "just now" in text or "today" in text:
        return base.date()

    for pattern, unit, days_per_unit in _RELATIVE_OFFSETS:
        m = pattern.search(text)
        if m:
            n = int(m.group(1))
            delta = timedelta(days=0 if unit == "hours" else n * days_per_unit)
            return (base - delta).date()

    return None

In [10]:
def _compute_posted_time(posted_at: str, crawled_dt) -> Optional[datetime]:
    if crawled_dt is None:
        return None
    text = (posted_at or "").lower()
    if "just now" in text or "today" in text:
        return crawled_dt
    for pattern, unit, days_per_unit in _RELATIVE_OFFSETS:
        m = pattern.search(text)
        if m:
            n = int(m.group(1))
            if unit == "hours":
                delta = timedelta(hours=n)
            else:
                delta = timedelta(days=n * days_per_unit)
            return crawled_dt - delta
    return None


class JobClassification(BaseModel):
    skills:        List[str] = Field(default_factory=list)
    job_expertise: List[str] = Field(default_factory=list)
    job_domains:   List[str] = Field(default_factory=list)


class JobContent(BaseModel):
    title:                str       = Field(..., min_length=1)
    work_mode:            str       = Field("")
    description:          str       = Field("")
    requirements:         str       = Field("")
    benefits_description: str       = Field("")
    top_reasons:          List[str] = Field(default_factory=list)


class CompanyInfo(BaseModel):
    company_name:      str = ""
    company_full_name: str = ""
    type:              str = ""
    industry:          str = ""
    size:              str = ""
    country:           str = ""
    working_days:      str = ""
    overtime_policy:   str = ""

print("Sub-models defined: JobClassification, JobContent, CompanyInfo")

Sub-models defined: JobClassification, JobContent, CompanyInfo


In [11]:
class JobListing(BaseModel):
    job_id:          Optional[str]      = Field(None)
    job_url:         str                = Field(...)
    posted_datetime: Optional[datetime] = Field(None)   
    location:        str                = Field("")      

    jobClassification:  JobClassification
    jobContent:         JobContent
    company:            CompanyInfo

    @field_validator("job_url")
    @classmethod
    def url_must_be_valid(cls, v: str) -> str:
        if not v.startswith("http"):
            raise ValueError(f"Invalid URL: {v!r}")
        return v

    @classmethod
    def from_raw(cls, raw: dict) -> "JobListing":
        job_url    = raw.get("job_url", "")
        posted_at  = raw.get("posted_at", "")
        crawled_at = raw.get("crawled_at", "")

        try:
            crawled_dt = datetime.strptime(crawled_at, "%Y-%m-%d %H:%M:%S")
        except (ValueError, TypeError):
            crawled_dt = None

        return cls(
            job_id          = _extract_job_id(job_url),
            job_url         = job_url,
            posted_datetime = _compute_posted_time(posted_at, crawled_dt),  
            location        = raw.get("location", ""),
            jobClassification  = JobClassification(
                skills        = raw.get("skills", []),
                job_expertise = raw.get("job_expertise", []),
                job_domains   = raw.get("job_domains", []),
            ),
            jobContent = JobContent(
                title                = raw.get("title", ""),
                work_mode            = raw.get("work_mode", ""),
                description          = raw.get("description", ""),
                requirements         = raw.get("requirements", ""),
                benefits_description = raw.get("benefits_description", ""),
                top_reasons          = raw.get("top_reasons", []),
            ),
            company = CompanyInfo(
                company_name      = raw.get("company_name", ""),
                company_full_name = raw.get("company_full_name", ""),
                type              = raw.get("type", ""),
                industry          = raw.get("industry", ""),
                size              = raw.get("size", ""),
                country           = raw.get("country", ""),
                working_days      = raw.get("working_days", ""),
                overtime_policy   = raw.get("overtime_policy", ""),
            ),
        )

print("Pydantic schema defined")

Pydantic schema defined


In [12]:
validated_listings: list[JobListing] = []
validation_errors:  list[dict]       = []

for i, raw in enumerate(raw_items):
    try:
        listing = JobListing.from_raw(raw)
        validated_listings.append(listing)
    except Exception as exc:
        validation_errors.append({
            "index": i,
            "url": raw.get("job_url"),
            "error": str(exc),
        })

print(f"Valid   : {len(validated_listings)}")
print(f"Invalid : {len(validation_errors)}")

if validation_errors:
    print("\nFailed items:")
    for e in validation_errors:
        print(f"  [{e['index']}] {e['url']}\n       {e['error']}")

Valid   : 300
Invalid : 0


# Export to file

In [13]:
OUTPUT_PATH = "itviec_jobs.json"

output = [
    listing.model_dump(mode="json")
        for listing in validated_listings
]

with open(OUTPUT_PATH, "w", encoding="utf-8") as fh:
    json.dump(output, fh, ensure_ascii=False, indent=2)

print(f"Exported {len(validated_listings)} listings → {OUTPUT_PATH}")

Exported 300 listings → itviec_jobs.json
